# Telecom Customer Churn — Data Preparation

## Objective

This notebook prepares the raw telecom customer dataset for machine-learning
experiments while preserving the behavioural and temporal information identified
during the data-understanding phase.

The preparation process focuses on:

- identifying and removing non-informative variables,
- resolving redundant representations,
- handling structurally missing telecom data,
- constructing meaningful temporal and behavioural features,
- preventing information leakage,
- and producing a reproducible model-ready dataset.

No machine-learning model is trained in this notebook.

The original customer-level observations are retained as the foundation,
while additional features are derived to represent behavioural change,
recency and persistence across the observed monthly period.

In [2]:
# Core libraries required for data preparation and feature engineering.

import numpy as np
import pandas as pd

from pathlib import Path

In [3]:
# Define the raw dataset location relative to the notebook directory.

DATA_PATH = (
    Path("../data/raw")
    / "telecom-churn-case-study-hackathon-C33"
    / "train.csv"
)

print(f"Dataset path: {DATA_PATH.resolve()}")

df = pd.read_csv(DATA_PATH)

print("\nDataset loaded successfully.")
print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")

Dataset path: E:\Churn Prediction\data\raw\telecom-churn-case-study-hackathon-C33\train.csv

Dataset loaded successfully.
Rows    : 69999
Columns : 172


## 3. Feature Audit

Before modifying the dataset, each feature is audited according to its
information content, role, redundancy, missingness, and temporal relevance.

The purpose of this step is to distinguish:

- identifiers,
- target variables,
- metadata,
- constant or near-constant variables,
- duplicate variables,
- behavioural variables,
- temporal variables,
- and variables requiring special missing-value treatment.

No features are removed or imputed during this audit.

In [5]:
# Create a compact feature audit table before any transformation is applied.

feature_audit = pd.DataFrame({
    "feature": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "missing_count": df.isna().sum().values,
    "missing_pct": (df.isna().mean() * 100).values,
    "unique_count": df.nunique(dropna=False).values
})

feature_audit["role"] = "To be determined"

print(f"Total features audited: {len(feature_audit)}")
display(feature_audit)

Total features audited: 172


,feature,dtype,missing_count,missing_pct,unique_count,role
0,id,int64,0,0.000000,69999,To be determined
1,circle_id,int64,0,0.000000,1,To be determined
2,loc_og_t2o_mou,float64,702,1.002871,2,To be determined
3,std_og_t2o_mou,float64,702,1.002871,2,To be determined
4,loc_ic_t2o_mou,float64,702,1.002871,2,To be determined
...,...,...,...,...,...,...
167,aon,int64,0,0.000000,3455,To be determined
168,aug_vbc_3g,float64,0,0.000000,10609,To be determined
169,jul_vbc_3g,float64,0,0.000000,10257,To be determined
170,jun_vbc_3g,float64,0,0.000000,9617,To be determined


In [6]:
# Inspect features with very high missingness.
# These will be investigated rather than automatically removed.

high_missing = (
    feature_audit[
        feature_audit["missing_pct"] >= 50
    ]
    .sort_values("missing_pct", ascending=False)
)

print(f"Features with >=50% missing values: {len(high_missing)}")
display(high_missing)

Features with >=50% missing values: 30


,feature,dtype,missing_count,missing_pct,unique_count,role
119,date_of_last_rech_data_6,str,52431,74.902499,31,To be determined
122,total_rech_data_6,float64,52431,74.902499,37,To be determined
128,count_rech_2g_6,float64,52431,74.902499,31,To be determined
131,count_rech_3g_6,float64,52431,74.902499,24,To be determined
125,max_rech_data_6,float64,52431,74.902499,48,To be determined
134,av_rech_amt_data_6,float64,52431,74.902499,793,To be determined
149,night_pck_user_6,float64,52431,74.902499,3,To be determined
146,arpu_2g_6,float64,52431,74.902499,5390,To be determined
143,arpu_3g_6,float64,52431,74.902499,5507,To be determined
164,fb_user_6,float64,52431,74.902499,3,To be determined


In [7]:
# Identify constant and near-constant features.
# A feature occurring with only one value provides no discriminatory information.

constant_features = [
    col for col in df.columns
    if df[col].nunique(dropna=False) <= 1
]

near_constant_features = []

for col in df.columns:
    value_share = df[col].value_counts(dropna=False, normalize=True)
    
    if len(value_share) > 0 and value_share.iloc[0] >= 0.99:
        near_constant_features.append(col)

print("Constant features:")
print(constant_features)

print("\nNear-constant features (>=99% one value):")
print(near_constant_features)

Constant features:
['circle_id', 'last_date_of_month_6']

Near-constant features (>=99% one value):
['circle_id', 'last_date_of_month_6', 'last_date_of_month_7']


In [8]:
# Identify exact duplicate columns.
# Duplicate representations should be verified semantically before removal.

duplicate_features = []

columns = df.columns

for i in range(len(columns)):
    for j in range(i + 1, len(columns)):
        col_a = columns[i]
        col_b = columns[j]

        if df[col_a].equals(df[col_b]):
            duplicate_features.append((col_a, col_b))

print(f"Exact duplicate feature pairs: {len(duplicate_features)}")

for pair in duplicate_features:
    print(pair)

Exact duplicate feature pairs: 6
('loc_og_t2o_mou', 'std_og_t2o_mou')
('loc_og_t2o_mou', 'loc_ic_t2o_mou')
('std_og_t2o_mou', 'loc_ic_t2o_mou')
('std_og_t2c_mou_6', 'std_ic_t2o_mou_6')
('std_og_t2c_mou_7', 'std_ic_t2o_mou_7')
('std_og_t2c_mou_8', 'std_ic_t2o_mou_8')


## 3.1 Feature Audit Decisions

The initial audit identified several categories of variables requiring
different preprocessing treatment.

### Identifiers

`id` is retained for customer-level traceability but excluded from the
machine-learning feature matrix.

### Target

`churn_probability` is retained as the target variable and excluded from
predictor features.

### Constant and Metadata Variables

`circle_id` is constant across the dataset and therefore contains no
discriminative information.

`last_date_of_month_6`, `last_date_of_month_7`, and
`last_date_of_month_8` represent observation-period metadata rather than
customer-specific behaviour. These variables are therefore excluded from
the predictive feature set.

### Behavioural Variables

Monthly usage, recharge, ARPU, data consumption, roaming and related
variables are retained because they describe customer behaviour across
the observed period.

### Recharge-Date Variables

`date_of_last_rech_*` and `date_of_last_rech_data_*` are retained for
temporal feature engineering because they may provide customer-level
recency information.

### High-Missingness Variables

High missingness, particularly among data-recharge variables, is not
treated as sufficient evidence for feature removal. Earlier investigation
indicated that missing data-recharge observations may correspond to the
absence of data-recharge activity.

The missingness mechanism will therefore be investigated before selecting
an imputation or representation strategy.

### Duplicate Variables

Exact duplicate feature groups have been identified. Their semantic
definitions will be checked against the data dictionary before redundant
columns are removed.

In [4]:
# Features identified as non-predictive metadata or constant variables.

metadata_features = [
    "circle_id",
    "last_date_of_month_6",
    "last_date_of_month_7",
    "last_date_of_month_8"
]

target = "churn_probability"
customer_id = "id"

print("Target:", target)
print("Customer identifier:", customer_id)

print("\nMetadata features scheduled for removal:")
for feature in metadata_features:
    print("-", feature)

Target: churn_probability
Customer identifier: id

Metadata features scheduled for removal:
- circle_id
- last_date_of_month_6
- last_date_of_month_7
- last_date_of_month_8


In [5]:
# Create a working dataset while preserving the original raw dataframe.
# The customer identifier and target remain available for downstream use.

df_prepared = df.drop(columns=metadata_features).copy()

print(f"Original columns : {df.shape[1]}")
print(f"Working columns  : {df_prepared.shape[1]}")

Original columns : 172
Working columns  : 168


In [6]:
# Inspect the data dictionary to verify the meaning of the
# feature groups identified as exact duplicates.

DICTIONARY_PATH = (
    Path("../data/raw")
    / "telecom-churn-case-study-hackathon-C33"
    / "data_dictionary.csv"
)

data_dictionary = pd.read_csv(DICTIONARY_PATH)

print(f"Dictionary shape: {data_dictionary.shape}")
display(data_dictionary.head(20))

Dictionary shape: (36, 2)


,Acronyms,Description
0,CIRCLE_ID,Telecom circle area to which the customer belo...
1,LOC,Local calls within same telecom circle
2,STD,STD calls outside the calling circle
3,IC,Incoming calls
4,OG,Outgoing calls
5,T2T,Operator T to T ie within same operator mobile...
6,T2M,Operator T to other operator mobile
7,T2O,Operator T to other operator fixed line
8,T2F,Operator T to fixed lines of T
9,T2C,Operator T to its own call center


In [8]:
# Inspect the dictionary structure and column names before filtering.

print("Dictionary columns:")
print(data_dictionary.columns.tolist())

Dictionary columns:
['Acronyms', 'Description']


## 4.1 Semantic Validation of Duplicate Features

The data dictionary was reviewed to interpret the telecom abbreviations used
in the feature names.

The dictionary defines dimensions such as:

- LOC — local calls within the same telecom circle
- STD — calls outside the calling circle
- IC — incoming calls
- OG — outgoing calls
- T2T — operator to same operator
- T2M — operator to another operator mobile
- T2O — operator to other operator/fixed-line category
- T2F — operator to fixed lines
- T2C — operator to call center
- MOU — minutes of usage
- RECH — recharge
- AON — age on network

Several feature pairs were found to contain exactly identical values.
However, identical numerical values do not necessarily imply identical
business meaning.

For example, features such as `loc_og_t2o_mou`, `std_og_t2o_mou`, and
`loc_ic_t2o_mou` represent different combinations of call direction and
call category even though their observed values are identical.

Therefore, these features are not removed solely because they are exact
duplicates. Their semantic role is retained during the preparation phase.

The same principle is applied to the monthly duplicate pairs such as
`std_og_t2c_mou_*` and `std_ic_t2o_mou_*`.

Feature reduction based on statistical redundancy can be considered later
during model feature selection, after the behavioural feature representation
has been constructed.

This prevents premature removal of variables that may be useful for
interpreting customer behaviour and deterioration.

In [9]:
# Compare missingness in data-recharge fields with observed data usage.
# This helps determine whether missing values represent structural absence
# of a service/activity rather than random data loss.

data_recharge_check = pd.DataFrame({
    "total_rech_data_6_missing": df["total_rech_data_6"].isna(),
    "vol_2g_mb_6": df["vol_2g_mb_6"],
    "vol_3g_mb_6": df["vol_3g_mb_6"],
    "total_rech_amt_6": df["total_rech_amt_6"],
    "arpu_6": df["arpu_6"]
})

display(
    data_recharge_check
    .groupby("total_rech_data_6_missing")
    .agg({
        "vol_2g_mb_6": ["count", "mean", "max"],
        "vol_3g_mb_6": ["count", "mean", "max"],
        "total_rech_amt_6": ["count", "mean"],
        "arpu_6": ["count", "mean"]
    })
)

vol_2g_mb_6                     vol_3g_mb_6  \
                                count       mean      max       count   
total_rech_data_6_missing                                               
False                           17568  206.29115  10285.9       17568   
True                            52431    0.00000      0.0       52431   

                                               total_rech_amt_6              \
                                 mean      max            count        mean   
total_rech_data_6_missing                                                     
False                      486.789024  45735.4            17568  469.039333   
True                         0.000000      0.0            52431  280.928725   

                          arpu_6              
                           count        mean  
total_rech_data_6_missing                     
False                      17568  403.674653  
True                       52431  242.745057

In [10]:
# Verify whether missing data-recharge records consistently correspond
# to zero data usage across all observed months.

for month in ["6", "7", "8"]:
    
    check = pd.DataFrame({
        "data_recharge_missing": df[f"total_rech_data_{month}"].isna(),
        "vol_2g_mb": df[f"vol_2g_mb_{month}"],
        "vol_3g_mb": df[f"vol_3g_mb_{month}"]
    })
    
    print(f"\n===== MONTH {month} =====")
    
    display(
        check
        .groupby("data_recharge_missing")
        .agg({
            "vol_2g_mb": ["count", "mean", "max"],
            "vol_3g_mb": ["count", "mean", "max"]
        })
    )


===== MONTH 6 =====


vol_2g_mb                     vol_3g_mb              \
                          count       mean      max     count        mean   
data_recharge_missing                                                       
False                     17568  206.29115  10285.9     17568  486.789024   
True                      52431    0.00000      0.0     52431    0.000000   

                                
                           max  
data_recharge_missing           
False                  45735.4  
True                       0.0


===== MONTH 7 =====


vol_2g_mb                      vol_3g_mb              \
                          count        mean      max     count        mean   
data_recharge_missing                                                        
False                     17865  200.770391  7873.55     17865  505.193515   
True                      52134    0.000000     0.00     52134    0.000000   

                                 
                            max  
data_recharge_missing            
False                  28144.12  
True                       0.00


===== MONTH 8 =====


vol_2g_mb                       vol_3g_mb              \
                          count        mean       max     count        mean   
data_recharge_missing                                                         
False                     18417  190.523717  11117.61     18417  514.954791   
True                      51582    0.000000      0.00     51582    0.000000   

                                 
                            max  
data_recharge_missing            
False                  30036.06  
True                       0.00

## 5. Missingness Mechanism Analysis

The high-missingness data-recharge variables were investigated to determine
whether the missing values represent data-quality problems or structural
absence of service activity.

For each observed month (June, July and August), customers were grouped
according to whether `total_rech_data` was missing.

The analysis showed that:

- when `total_rech_data` was observed, 2G and 3G usage was also observed;
- when `total_rech_data` was missing, both `vol_2g_mb` and `vol_3g_mb`
  were exactly zero for all affected customers;
- the same pattern was observed consistently across all three months.

This indicates that the missingness in the data-recharge feature group is
strongly associated with absence of data-service activity rather than random
loss of observations.

Therefore, these missing values are treated as **structural missingness**.

### Preparation decision

For numerical data-service activity, recharge and usage variables where
absence of activity has a natural value of zero, missing values will be
represented as zero.

Date variables will not be converted to zero. Their missingness will instead
be interpreted as absence of the corresponding recharge event and used when
constructing activity or recency features.

No global `fillna(0)` operation will be applied to the dataset because the
meaning of missingness depends on the business semantics of each feature.

This approach preserves the observed behavioural trajectory of customers
while avoiding the assumption that every missing value has the same meaning.

In [11]:
# Identify data-service variables using the feature naming structure.
# The selected columns will be reviewed before any imputation is applied.

data_service_keywords = [
    "total_rech_data",
    "max_rech_data",
    "count_rech_2g",
    "count_rech_3g",
    "av_rech_amt_data",
    "vol_2g_mb",
    "vol_3g_mb",
    "arpu_2g",
    "arpu_3g",
    "monthly_2g",
    "sachet_2g",
    "monthly_3g",
    "sachet_3g",
    "vbc_3g"
]

data_service_features = [
    col
    for col in df_prepared.columns
    if any(keyword in col for keyword in data_service_keywords)
]

print(f"Data-service numerical features identified: {len(data_service_features)}")

for feature in data_service_features:
    print("-", feature)

Data-service numerical features identified: 42
- total_rech_data_6
- total_rech_data_7
- total_rech_data_8
- max_rech_data_6
- max_rech_data_7
- max_rech_data_8
- count_rech_2g_6
- count_rech_2g_7
- count_rech_2g_8
- count_rech_3g_6
- count_rech_3g_7
- count_rech_3g_8
- av_rech_amt_data_6
- av_rech_amt_data_7
- av_rech_amt_data_8
- vol_2g_mb_6
- vol_2g_mb_7
- vol_2g_mb_8
- vol_3g_mb_6
- vol_3g_mb_7
- vol_3g_mb_8
- arpu_3g_6
- arpu_3g_7
- arpu_3g_8
- arpu_2g_6
- arpu_2g_7
- arpu_2g_8
- monthly_2g_6
- monthly_2g_7
- monthly_2g_8
- sachet_2g_6
- sachet_2g_7
- sachet_2g_8
- monthly_3g_6
- monthly_3g_7
- monthly_3g_8
- sachet_3g_6
- sachet_3g_7
- sachet_3g_8
- aug_vbc_3g
- jul_vbc_3g
- jun_vbc_3g


In [12]:
# Review missingness within the identified data-service feature group.
# This separates structural data-recharge fields from other data-related variables.

data_service_missing = (
    df_prepared[data_service_features]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .to_frame("missing_pct")
)

display(data_service_missing)

,missing_pct
total_rech_data_6,74.902499
max_rech_data_6,74.902499
count_rech_3g_6,74.902499
count_rech_2g_6,74.902499
arpu_3g_6,74.902499
av_rech_amt_data_6,74.902499
arpu_2g_6,74.902499
count_rech_2g_7,74.478207
total_rech_data_7,74.478207
max_rech_data_7,74.478207


## 5. Structural Missingness Treatment

The missing-value analysis identified a distinct pattern within the
data-recharge feature group.

The following features contain approximately 74% missing values across the
three observed months:

* `total_rech_data`
* `max_rech_data`
* `count_rech_2g`
* `count_rech_3g`
* `arpu_2g`
* `arpu_3g`
* `av_rech_amt_data`

The missingness was compared against corresponding mobile-data usage
variables. Customers for whom `total_rech_data` was missing consistently
showed zero values for both 2G and 3G data usage in the corresponding month.

This indicates that the missing values are strongly associated with the
absence of data-recharge activity rather than being arbitrary missing
observations.

Therefore, these data-recharge numerical variables are treated as
structural missingness and converted to zero.

This transformation is applied only to variables for which zero has a valid
business interpretation of no recorded activity.

Variables that are already fully observed are not modified, and date
variables are not zero-imputed because a missing date represents the absence
of an event rather than a numerical quantity.

A global missing-value replacement strategy is deliberately avoided.


In [13]:
# Data-recharge numerical features for which missingness represents
# absence of the corresponding data-recharge activity.

data_recharge_features = [
    col
    for col in df_prepared.columns
    if any(
        keyword in col
        for keyword in [
            "total_rech_data",
            "max_rech_data",
            "count_rech_2g",
            "count_rech_3g",
            "arpu_2g",
            "arpu_3g",
            "av_rech_amt_data"
        ]
    )
]

print(f"Features selected for structural zero-imputation: "
      f"{len(data_recharge_features)}")

for feature in data_recharge_features:
    print("-", feature)

Features selected for structural zero-imputation: 21
- total_rech_data_6
- total_rech_data_7
- total_rech_data_8
- max_rech_data_6
- max_rech_data_7
- max_rech_data_8
- count_rech_2g_6
- count_rech_2g_7
- count_rech_2g_8
- count_rech_3g_6
- count_rech_3g_7
- count_rech_3g_8
- av_rech_amt_data_6
- av_rech_amt_data_7
- av_rech_amt_data_8
- arpu_3g_6
- arpu_3g_7
- arpu_3g_8
- arpu_2g_6
- arpu_2g_7
- arpu_2g_8


In [14]:
# Preserve the missing-value counts before applying the transformation
# so that the effect of the imputation can be verified.

missing_before = df_prepared[data_recharge_features].isna().sum()

# Missing values in these variables represent absence of data-recharge
# activity and are therefore represented explicitly as zero.

df_prepared[data_recharge_features] = (
    df_prepared[data_recharge_features]
    .fillna(0)
)

missing_after = df_prepared[data_recharge_features].isna().sum()

missing_check = pd.DataFrame({
    "missing_before": missing_before,
    "missing_after": missing_after
})

display(missing_check)

,missing_before,missing_after
total_rech_data_6,52431,0
total_rech_data_7,52134,0
total_rech_data_8,51582,0
max_rech_data_6,52431,0
max_rech_data_7,52134,0
max_rech_data_8,51582,0
count_rech_2g_6,52431,0
count_rech_2g_7,52134,0
count_rech_2g_8,51582,0
count_rech_3g_6,52431,0


In [15]:
# Check which features still contain missing values after
# structural data-recharge treatment.

remaining_missing = (
    df_prepared
    .isna()
    .sum()
    .sort_values(ascending=False)
)

remaining_missing = remaining_missing[
    remaining_missing > 0
]

print(f"Features still containing missing values: "
      f"{len(remaining_missing)}")

display(remaining_missing)

Features still containing missing values: 102


fb_user_6                   52431
date_of_last_rech_data_6    52431
night_pck_user_6            52431
date_of_last_rech_data_7    52134
night_pck_user_7            52134
                            ...  
date_of_last_rech_7          1234
date_of_last_rech_6          1101
std_og_t2o_mou                702
loc_ic_t2o_mou                702
loc_og_t2o_mou                702
Length: 102, dtype: int64

In [16]:
# Check which features still contain missing values after
# structural data-recharge treatment.

remaining_missing = (
    df_prepared
    .isna()
    .sum()
    .sort_values(ascending=False)
)

remaining_missing = remaining_missing[
    remaining_missing > 0
]

print(f"Features still containing missing values: "
      f"{len(remaining_missing)}")

display(remaining_missing)

Features still containing missing values: 102


fb_user_6                   52431
date_of_last_rech_data_6    52431
night_pck_user_6            52431
date_of_last_rech_data_7    52134
night_pck_user_7            52134
                            ...  
date_of_last_rech_7          1234
date_of_last_rech_6          1101
std_og_t2o_mou                702
loc_ic_t2o_mou                702
loc_og_t2o_mou                702
Length: 102, dtype: int64

In [18]:
# Verify whether the missingness of data-service status/date variables
# exactly follows the structural data-recharge missingness pattern.

structural_data_features = [
    "total_rech_data_6",
    "fb_user_6",
    "night_pck_user_6",
    "date_of_last_rech_data_6",
    
    "total_rech_data_7",
    "fb_user_7",
    "night_pck_user_7",
    "date_of_last_rech_data_7",
    
    "total_rech_data_8",
    "fb_user_8",
    "night_pck_user_8",
    "date_of_last_rech_data_8"
]

structural_missing_check = pd.DataFrame({
    feature: df_prepared[feature].isna()
    for feature in structural_data_features
})

display(
    structural_missing_check.sum()
)

total_rech_data_6               0
fb_user_6                   52431
night_pck_user_6            52431
date_of_last_rech_data_6    52431
total_rech_data_7               0
fb_user_7                   52134
night_pck_user_7            52134
date_of_last_rech_data_7    52134
total_rech_data_8               0
fb_user_8                   51582
night_pck_user_8            51582
date_of_last_rech_data_8    51582
dtype: int64

In [19]:
# Verify whether the missingness masks are identical rather than merely
# having the same number of missing observations.

for month in ["6", "7", "8"]:
    
    reference = df_prepared[f"total_rech_data_{month}"].isna()
    
    print(f"\nMonth {month}")
    
    for feature in [
        f"fb_user_{month}",
        f"night_pck_user_{month}",
        f"date_of_last_rech_data_{month}"
    ]:
        same_pattern = (
            reference.equals(df_prepared[feature].isna())
        )
        
        print(f"{feature}: {same_pattern}")


Month 6
fb_user_6: False
night_pck_user_6: False
date_of_last_rech_data_6: False

Month 7
fb_user_7: False
night_pck_user_7: False
date_of_last_rech_data_7: False

Month 8
fb_user_8: False
night_pck_user_8: False
date_of_last_rech_data_8: False


In [20]:
for month in ["6", "7", "8"]:

    print(f"\n========== MONTH {month} ==========")

    reference = df_prepared[f"total_rech_data_{month}"].isna()

    for feature in [
        f"fb_user_{month}",
        f"night_pck_user_{month}",
        f"date_of_last_rech_data_{month}"
    ]:

        current = df_prepared[feature].isna()

        both_missing = (reference & current).sum()
        total_rech_only = (reference & ~current).sum()
        feature_only = (~reference & current).sum()

        print(f"\n{feature}")
        print(f"Both missing       : {both_missing}")
        print(f"Total rech only    : {total_rech_only}")
        print(f"{feature} only      : {feature_only}")


========== MONTH 6 ==========

fb_user_6
Both missing       : 0
Total rech only    : 0
fb_user_6 only      : 52431

night_pck_user_6
Both missing       : 0
Total rech only    : 0
night_pck_user_6 only      : 52431

date_of_last_rech_data_6
Both missing       : 0
Total rech only    : 0
date_of_last_rech_data_6 only      : 52431

========== MONTH 7 ==========

fb_user_7
Both missing       : 0
Total rech only    : 0
fb_user_7 only      : 52134

night_pck_user_7
Both missing       : 0
Total rech only    : 0
night_pck_user_7 only      : 52134

date_of_last_rech_data_7
Both missing       : 0
Total rech only    : 0
date_of_last_rech_data_7 only      : 52134

========== MONTH 8 ==========

fb_user_8
Both missing       : 0
Total rech only    : 0
fb_user_8 only      : 51582

night_pck_user_8
Both missing       : 0
Total rech only    : 0
night_pck_user_8 only      : 51582

date_of_last_rech_data_8
Both missing       : 0
Total rech only    : 0
date_of_last_rech_data_8 only      : 51582


In [21]:
for feature in [
    "fb_user_6",
    "fb_user_7",
    "fb_user_8",
    "night_pck_user_6",
    "night_pck_user_7",
    "night_pck_user_8"
]:

    print(f"\n{feature}")
    print(df_prepared[feature].value_counts(dropna=False))


fb_user_6
fb_user_6
NaN    52431
1.0    16098
0.0     1470
Name: count, dtype: int64

fb_user_7
fb_user_7
NaN    52134
1.0    16249
0.0     1616
Name: count, dtype: int64

fb_user_8
fb_user_8
NaN    51582
1.0    16397
0.0     2020
Name: count, dtype: int64

night_pck_user_6
night_pck_user_6
NaN    52431
0.0    17124
1.0      444
Name: count, dtype: int64

night_pck_user_7
night_pck_user_7
NaN    52134
0.0    17435
1.0      430
Name: count, dtype: int64

night_pck_user_8
night_pck_user_8
NaN    51582
0.0    18030
1.0      387
Name: count, dtype: int64


In [22]:
check_cols = [
    "total_rech_data_6",
    "fb_user_6",
    "night_pck_user_6",
    "date_of_last_rech_data_6",
    "vol_2g_mb_6",
    "vol_3g_mb_6",
    "monthly_2g_6",
    "monthly_3g_6",
    "sachet_2g_6",
    "sachet_3g_6"
]

display(
    df_prepared[check_cols]
    .isna()
    .corr()
)

,total_rech_data_6,fb_user_6,night_pck_user_6,date_of_last_rech_data_6,vol_2g_mb_6,vol_3g_mb_6,monthly_2g_6,monthly_3g_6,sachet_2g_6,sachet_3g_6
total_rech_data_6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
fb_user_6,NaN,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN
night_pck_user_6,NaN,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN
date_of_last_rech_data_6,NaN,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN
vol_2g_mb_6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
vol_3g_mb_6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
monthly_2g_6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
monthly_3g_6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sachet_2g_6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sachet_3g_6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [23]:
# Preserve the original missingness pattern from the raw dataset.
# This is required because df_prepared has already undergone structural imputation.

original_data_missingness = pd.DataFrame({
    "total_rech_data_6": df["total_rech_data_6"].isna(),
    "fb_user_6": df["fb_user_6"].isna(),
    "night_pck_user_6": df["night_pck_user_6"].isna(),
    "date_of_last_rech_data_6": df["date_of_last_rech_data_6"].isna(),

    "total_rech_data_7": df["total_rech_data_7"].isna(),
    "fb_user_7": df["fb_user_7"].isna(),
    "night_pck_user_7": df["night_pck_user_7"].isna(),
    "date_of_last_rech_data_7": df["date_of_last_rech_data_7"].isna(),

    "total_rech_data_8": df["total_rech_data_8"].isna(),
    "fb_user_8": df["fb_user_8"].isna(),
    "night_pck_user_8": df["night_pck_user_8"].isna(),
    "date_of_last_rech_data_8": df["date_of_last_rech_data_8"].isna()
})

display(original_data_missingness.sum())

total_rech_data_6           52431
fb_user_6                   52431
night_pck_user_6            52431
date_of_last_rech_data_6    52431
total_rech_data_7           52134
fb_user_7                   52134
night_pck_user_7            52134
date_of_last_rech_data_7    52134
total_rech_data_8           51582
fb_user_8                   51582
night_pck_user_8            51582
date_of_last_rech_data_8    51582
dtype: int64

In [24]:
# Compare the original missingness pattern across data-service variables.
for month in ["6", "7", "8"]:

    print(f"\n========== MONTH {month} ==========")

    reference = df[f"total_rech_data_{month}"].isna()

    for feature in [
        f"fb_user_{month}",
        f"night_pck_user_{month}",
        f"date_of_last_rech_data_{month}"
    ]:

        current = df[feature].isna()

        print(
            f"{feature}: "
            f"{reference.equals(current)}"
        )


========== MONTH 6 ==========
fb_user_6: True
night_pck_user_6: True
date_of_last_rech_data_6: True

========== MONTH 7 ==========
fb_user_7: True
night_pck_user_7: True
date_of_last_rech_data_7: True

========== MONTH 8 ==========
fb_user_8: True
night_pck_user_8: True
date_of_last_rech_data_8: True


## Structural Missingness in Data-Service Variables

The raw dataset was examined to distinguish structural missingness from
ordinary incomplete observations.

The variables `total_rech_data_*`, `fb_user_*`, `night_pck_user_*`, and
`date_of_last_rech_data_*` exhibit the same missingness pattern across the
observed monthly periods.

This indicates that the missing values are associated with a common
data-service activity state rather than representing independent missing
observations.

Numerical data-recharge variables are therefore encoded as zero where the
absence of activity has a valid numerical interpretation.

Binary service indicators such as `fb_user_*` and `night_pck_user_*` are
handled separately from date variables.

The `date_of_last_rech_data_*` variables are not numerically imputed because
zero does not represent a meaningful date. Their missingness will instead be
used when constructing explicit data-service activity and recency features.

This approach preserves the behavioral meaning of the original data rather
than applying a blanket missing-value strategy.

In [25]:
# Binary data-service indicators use zero to represent the absence
# of the corresponding recorded service scheme.

binary_data_service_features = [
    "fb_user_6",
    "fb_user_7",
    "fb_user_8",
    "night_pck_user_6",
    "night_pck_user_7",
    "night_pck_user_8"
]

df_prepared[binary_data_service_features] = (
    df_prepared[binary_data_service_features]
    .fillna(0)
)

print("Binary data-service missing values after imputation:")

display(
    df_prepared[binary_data_service_features]
    .isna()
    .sum()
)

Binary data-service missing values after imputation:


fb_user_6           0
fb_user_7           0
fb_user_8           0
night_pck_user_6    0
night_pck_user_7    0
night_pck_user_8    0
dtype: int64

In [26]:
# Verify that binary service indicators no longer contain missing values.
# Date variables are intentionally excluded because their missingness
# represents the absence of a recorded data-recharge event.

remaining_binary_missing = (
    df_prepared[binary_data_service_features]
    .isna()
    .sum()
)

display(remaining_binary_missing)

fb_user_6           0
fb_user_7           0
fb_user_8           0
night_pck_user_6    0
night_pck_user_7    0
night_pck_user_8    0
dtype: int64

## Treatment of Structural Data-Service Missingness

The raw dataset was used to verify the original missingness pattern before
imputation.

For each observed month, `total_rech_data`, `fb_user`, `night_pck_user`, and
`date_of_last_rech_data` were found to have identical missingness masks.

This indicates that these variables represent a common data-service activity
state rather than independent randomly missing observations.

The numerical data-recharge variables have therefore been encoded as zero,
where zero has a valid interpretation as absence of recorded activity.

The binary service indicators `fb_user_*` and `night_pck_user_*` are also
encoded as zero when the corresponding structural state is absent.

The `date_of_last_rech_data_*` variables are intentionally not numerically
imputed. A missing date carries behavioral information indicating that no
corresponding data-recharge date was recorded. This information will be
converted into explicit activity and recency features during temporal feature
engineering.

This treatment preserves the semantic meaning of the original telecom data
instead of applying a generic missing-value imputation strategy.

In [27]:
# Investigate whether missing recharge dates correspond to the absence
# of recharge activity in the corresponding monthly observation.

recharge_date_check = pd.DataFrame({
    "date_missing_6": df["date_of_last_rech_6"].isna(),
    "total_rech_amt_6": df["total_rech_amt_6"],
    "total_rech_num_6": df["total_rech_num_6"],

    "date_missing_7": df["date_of_last_rech_7"].isna(),
    "total_rech_amt_7": df["total_rech_amt_7"],
    "total_rech_num_7": df["total_rech_num_7"],

    "date_missing_8": df["date_of_last_rech_8"].isna(),
    "total_rech_amt_8": df["total_rech_amt_8"],
    "total_rech_num_8": df["total_rech_num_8"]
})

display(recharge_date_check.head())

,date_missing_6,total_rech_amt_6,total_rech_num_6,date_missing_7,total_rech_amt_7,total_rech_num_7,date_missing_8,total_rech_amt_8,total_rech_num_8
0,False,77,3,False,65,2,False,10,2
1,False,0,3,False,145,4,False,50,5
2,False,70,2,False,120,4,False,0,2
3,False,160,2,False,240,4,False,130,3
4,False,290,13,False,136,10,False,122,8


In [28]:
# Compare recharge activity for customers with and without a recorded
# last-recharge date in June.

display(
    recharge_date_check
    .groupby("date_missing_6")
    .agg({
        "total_rech_amt_6": ["count", "mean", "min", "max"],
        "total_rech_num_6": ["count", "mean", "min", "max"]
    })
)

total_rech_amt_6                        total_rech_num_6  \
                          count        mean min    max            count   
date_missing_6                                                            
False                     68898  333.383509   0  35190            68898   
True                       1101    0.000000   0      0             1101   

                                   
                    mean min  max  
date_missing_6                     
False           7.687437   1  170  
True            0.000000   0    0

In [29]:
# Verify whether missing recharge dates consistently represent
# the absence of recharge activity across all observed months.

for month in ["6", "7", "8"]:

    date_col = f"date_of_last_rech_{month}"
    amount_col = f"total_rech_amt_{month}"
    count_col = f"total_rech_num_{month}"

    print(f"\n{'=' * 60}")
    print(f"Month: {month}")
    print(f"{'=' * 60}")

    display(
        df.groupby(df[date_col].isna())
        .agg({
            amount_col: ["count", "mean", "min", "max"],
            count_col: ["count", "mean", "min", "max"]
        })
    )


Month: 6


total_rech_amt_6                        total_rech_num_6  \
                               count        mean min    max            count   
date_of_last_rech_6                                                            
False                          68898  333.383509   0  35190            68898   
True                            1101    0.000000   0      0             1101   

                                        
                         mean min  max  
date_of_last_rech_6                     
False                7.687437   1  170  
True                 0.000000   0    0


Month: 7


total_rech_amt_7                        total_rech_num_7  \
                               count        mean min    max            count   
date_of_last_rech_7                                                            
False                          68765  328.161463   0  40335            68765   
True                            1234    0.000000   0      0             1234   

                                        
                         mean min  max  
date_of_last_rech_7                     
False                7.844965   1  138  
True                 0.000000   0    0


Month: 8


total_rech_amt_8                       total_rech_num_8  \
                               count       mean min    max            count   
date_of_last_rech_8                                                           
False                          67538  335.64691   0  45320            67538   
True                            2461    0.00000   0      0             2461   

                                        
                         mean min  max  
date_of_last_rech_8                     
False                7.488199   1  138  
True                 0.000000   0    0

## Treatment of Missing Recharge Dates

The recharge-date variables `date_of_last_rech_6`, `date_of_last_rech_7`,
and `date_of_last_rech_8` were investigated against the corresponding
monthly recharge activity variables.

For all three observed months, customers with a missing
`date_of_last_rech_*` had:

- `total_rech_amt_* = 0`
- `total_rech_num_* = 0`

This establishes that the missing recharge date represents the absence of
recorded recharge activity during that month rather than an unknown recharge
date.

The date variables are therefore not imputed with an artificial date or
numeric value.

Instead, their missingness is retained as a meaningful behavioral state.
During temporal feature engineering, explicit recharge-activity and
recharge-recency features will be derived from the monthly recharge
variables.

This preserves the behavioral meaning of the original telecom data and
avoids introducing artificial temporal information.

In [30]:
# Inspect the observed value distributions of the three base T2O fields.
# These variables describe operator/call-type relationships rather than
# customer activity during a specific month.

t2o_base_features = [
    "loc_og_t2o_mou",
    "std_og_t2o_mou",
    "loc_ic_t2o_mou"
]

for feature in t2o_base_features:
    print(f"\n{'=' * 60}")
    print(feature)
    print(f"{'=' * 60}")
    
    print("Missing:", df[feature].isna().sum())
    print("Unique values:")
    print(df[feature].value_counts(dropna=False))


loc_og_t2o_mou
Missing: 702
Unique values:
loc_og_t2o_mou
0.0    69297
NaN      702
Name: count, dtype: int64

std_og_t2o_mou
Missing: 702
Unique values:
std_og_t2o_mou
0.0    69297
NaN      702
Name: count, dtype: int64

loc_ic_t2o_mou
Missing: 702
Unique values:
loc_ic_t2o_mou
0.0    69297
NaN      702
Name: count, dtype: int64


In [31]:
# Test whether the three base T2O fields share the same missingness pattern.
# A common mask would indicate that the missingness is associated with a
# shared dataset-level condition rather than independent feature failures.

for i in range(len(t2o_base_features)):
    for j in range(i + 1, len(t2o_base_features)):
        
        feature_a = t2o_base_features[i]
        feature_b = t2o_base_features[j]
        
        same_mask = (
            df[feature_a].isna()
            .equals(df[feature_b].isna())
        )
        
        print(
            f"{feature_a} vs {feature_b}: "
            f"{same_mask}"
        )

loc_og_t2o_mou vs std_og_t2o_mou: True
loc_og_t2o_mou vs loc_ic_t2o_mou: True
std_og_t2o_mou vs loc_ic_t2o_mou: True


In [32]:
# Evaluate whether these fields contain meaningful variation.
# If the observed values are effectively constant, the missingness may not
# justify creating an additional behavioral representation.

t2o_summary = pd.DataFrame({
    "feature": t2o_base_features,
    "missing_count": [
        df[col].isna().sum()
        for col in t2o_base_features
    ],
    "missing_pct": [
        df[col].isna().mean() * 100
        for col in t2o_base_features
    ],
    "unique_non_missing": [
        df[col].dropna().nunique()
        for col in t2o_base_features
    ],
    "non_missing_min": [
        df[col].dropna().min()
        for col in t2o_base_features
    ],
    "non_missing_max": [
        df[col].dropna().max()
        for col in t2o_base_features
    ]
})

display(t2o_summary)

,feature,missing_count,missing_pct,unique_non_missing,non_missing_min,non_missing_max
0,loc_og_t2o_mou,702,1.002871,1,0.0,0.0
1,std_og_t2o_mou,702,1.002871,1,0.0,0.0
2,loc_ic_t2o_mou,702,1.002871,1,0.0,0.0


In [33]:
# Base T2O fields contain only zero among observed records, and all three
# variables share the same missingness pattern. Zero is therefore used as
# the consistent representation of the observed/default telecom state.

t2o_base_features = [
    "loc_og_t2o_mou",
    "std_og_t2o_mou",
    "loc_ic_t2o_mou"
]

df_prepared[t2o_base_features] = (
    df_prepared[t2o_base_features]
    .fillna(0)
)

# Confirm that the structural missingness has been resolved.
display(
    df_prepared[t2o_base_features]
    .isna()
    .sum()
)

loc_og_t2o_mou    0
std_og_t2o_mou    0
loc_ic_t2o_mou    0
dtype: int64

## Treatment of Base T2O Missingness

The base T2O variables `loc_og_t2o_mou`, `std_og_t2o_mou`, and
`loc_ic_t2o_mou` were investigated as a single feature group.

All three variables contained exactly 702 missing observations and shared
the same missingness mask.

Among all non-missing observations, each variable had only one unique value:
`0.0`.

Therefore, there is no observed non-zero variation that would be lost by
representing the missing observations as zero.

The missing values are consequently encoded as `0`, providing a consistent
representation of the only observed state of these fields.

These variables will not be treated as meaningful behavioral drivers during
temporal feature engineering because they contain no observed variation
beyond this zero state.

In [36]:
# These base T2O fields contain no observed variation beyond zero.
# Their shared missingness is therefore encoded as the same zero state.

t2o_base_features = [
    "loc_og_t2o_mou",
    "std_og_t2o_mou",
    "loc_ic_t2o_mou"
]

df_prepared[t2o_base_features] = (
    df_prepared[t2o_base_features].fillna(0)
)

print("Remaining missing values:")
display(df_prepared[t2o_base_features].isna().sum())

Remaining missing values:


loc_og_t2o_mou    0
std_og_t2o_mou    0
loc_ic_t2o_mou    0
dtype: int64

In [37]:
# Separate remaining missing values by their semantic role.
# Date fields retain missingness because absence of a date can represent
# absence of the corresponding activity.

date_features = [
    col for col in df_prepared.columns
    if col.startswith("date_of_last_rech")
]

remaining_missing = (
    df_prepared.isna().sum()
    .loc[lambda x: x > 0]
    .sort_values(ascending=False)
)

print(f"Remaining features with missing values: {len(remaining_missing)}")
display(remaining_missing)

Remaining features with missing values: 93


date_of_last_rech_data_6    52431
date_of_last_rech_data_7    52134
date_of_last_rech_data_8    51582
roam_ic_mou_8                3703
onnet_mou_8                  3703
                            ...  
spl_ic_mou_7                 2687
ic_others_7                  2687
date_of_last_rech_8          2461
date_of_last_rech_7          1234
date_of_last_rech_6          1101
Length: 93, dtype: int64

In [38]:
date_features = [
    "date_of_last_rech_6",
    "date_of_last_rech_7",
    "date_of_last_rech_8",
    "date_of_last_rech_data_6",
    "date_of_last_rech_data_7",
    "date_of_last_rech_data_8"
]

In [39]:
# Remaining non-date numeric fields contain limited missing observations.
# Median imputation preserves the central distribution without assigning
# an artificial "no activity" interpretation.

numeric_features = df_prepared.select_dtypes(
    include=[np.number]
).columns.tolist()

numeric_imputation_features = [
    col for col in numeric_features
    if col not in date_features + [target]
]

df_prepared[numeric_imputation_features] = (
    df_prepared[numeric_imputation_features]
    .fillna(
        df_prepared[numeric_imputation_features].median()
    )
)

In [ ]:
#Verify that the numeric imputation has resolved all remaining missing values.
remaining_missing = (
    df_prepared.isna().sum()  
    .loc[lambda x: x > 0]
    .sort_values(ascending=False)
)

print(f"Remaining features with missing values: {len(remaining_missing)}")
display(remaining_missing)

Remaining features with missing values: 6


date_of_last_rech_data_6    52431
date_of_last_rech_data_7    52134
date_of_last_rech_data_8    51582
date_of_last_rech_8          2461
date_of_last_rech_7          1234
date_of_last_rech_6          1101
dtype: int64

In [41]:
# Encode whether recharge activity was recorded in each observed month.
# The date itself remains missing when no recharge occurred.

for month in ["6", "7", "8"]:
    df_prepared[f"recharge_active_{month}"] = (
        df_prepared[f"date_of_last_rech_{month}"].notna().astype(int)
    )

    df_prepared[f"data_recharge_active_{month}"] = (
        df_prepared[f"date_of_last_rech_data_{month}"].notna().astype(int)
    )

C:\Users\disha\AppData\Local\Temp\ipykernel_26956\4018577888.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_prepared[f"recharge_active_{month}"] = (
C:\Users\disha\AppData\Local\Temp\ipykernel_26956\4018577888.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_prepared[f"data_recharge_active_{month}"] = (
C:\Users\disha\AppData\Local\Temp\ipykernel_26956\4018577888.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  C

In [42]:
# Verify that the new binary activity indicators have been created correctly.
activity_features = [
    f"{prefix}_{month}"
    for prefix in ["recharge_active", "data_recharge_active"]
    for month in ["6", "7", "8"]
]

display(df_prepared[activity_features].sum())

recharge_active_6         68898
recharge_active_7         68765
recharge_active_8         67538
data_recharge_active_6    17568
data_recharge_active_7    17865
data_recharge_active_8    18417
dtype: int64

In [43]:
# Raw recharge dates are retained only through their derived activity state.
# This avoids introducing artificial numeric values for missing dates.

df_prepared = df_prepared.drop(columns=date_features)

print("Remaining missing values:")
display(
    df_prepared.isna().sum().loc[lambda x: x > 0]
)

Remaining missing values:


Series([], dtype: int64)

In [45]:
print(f"Columns after removing raw date features: {df_prepared.shape[1]}")

remaining_missing = (
    df_prepared.isna().sum()
    .loc[lambda x: x > 0]
    .sort_values(ascending=False)
)

print(f"\nRemaining features with missing values: {len(remaining_missing)}")
display(remaining_missing)

Columns after removing raw date features: 168

Remaining features with missing values: 0


Series([], dtype: int64)

#Temporal & Behavioral Feature Engineering

In [47]:
# Construct temporal behavioural features for the core customer-value
# and engagement dimensions identified during data understanding.

temporal_metrics = {
    "arpu": ["arpu_6", "arpu_7", "arpu_8"],
    "recharge_amount": ["total_rech_amt_6", "total_rech_amt_7", "total_rech_amt_8"],
    "recharge_count": ["total_rech_num_6", "total_rech_num_7", "total_rech_num_8"],
    "outgoing_mou": ["total_og_mou_6", "total_og_mou_7", "total_og_mou_8"],
    "incoming_mou": ["total_ic_mou_6", "total_ic_mou_7", "total_ic_mou_8"]
}

for metric, columns in temporal_metrics.items():
    june, july, august = columns

    # Month-to-month behavioural movement
    df_prepared[f"{metric}_jun_jul_change"] = (
        df_prepared[july] - df_prepared[june]
    )

    df_prepared[f"{metric}_jul_aug_change"] = (
        df_prepared[august] - df_prepared[july]
    )

    # Overall movement across the observed period
    df_prepared[f"{metric}_jun_aug_change"] = (
        df_prepared[august] - df_prepared[june]
    )

    # Continuous decline across all three observed months
    df_prepared[f"{metric}_persistent_decline"] = (
        (df_prepared[june] > df_prepared[july]) &
        (df_prepared[july] > df_prepared[august])
    ).astype(int)

print(f"Columns after temporal feature engineering: {df_prepared.shape[1]}")

Columns after temporal feature engineering: 188


In [48]:
print(df_prepared.shape)

display(
    df_prepared[
        [
            "arpu_jun_jul_change",
            "arpu_jul_aug_change",
            "arpu_jun_aug_change",
            "arpu_persistent_decline"
        ]
    ].head()
)

(69999, 188)


,arpu_jun_jul_change,arpu_jul_aug_change,arpu_jun_aug_change,arpu_persistent_decline
0,55.732,-79.482,-23.750,0
1,122.787,-79.834,42.953,0
2,42.370,-103.176,-60.806,0
3,48.898,-94.165,-45.267,0
4,-112.517,-26.626,-139.143,1


In [49]:
# Count how many core behavioural dimensions experienced an overall
# deterioration between June and August.

deterioration_features = [
    "arpu_jun_aug_change",
    "recharge_amount_jun_aug_change",
    "recharge_count_jun_aug_change",
    "outgoing_mou_jun_aug_change",
    "incoming_mou_jun_aug_change"
]

df_prepared["coordinated_deterioration_count"] = (
    df_prepared[deterioration_features] < 0
).sum(axis=1)

print(
    df_prepared["coordinated_deterioration_count"]
    .value_counts()
    .sort_index()
)

coordinated_deterioration_count
0    16246
1    10642
2     8529
3     9071
4    10186
5    15325
Name: count, dtype: int64


C:\Users\disha\AppData\Local\Temp\ipykernel_26956\754941602.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_prepared["coordinated_deterioration_count"] = (


In [50]:
# Rebuild the DataFrame into a contiguous representation after
# repeated feature construction.

df_prepared = df_prepared.copy()

print("DataFrame structure refreshed.")
print(f"Shape: {df_prepared.shape}")

DataFrame structure refreshed.
Shape: (69999, 189)


In [51]:
# Compare churn behaviour across the number of deteriorating
# behavioural dimensions.

coordinated_deterioration_analysis = (
    df_prepared
    .groupby("coordinated_deterioration_count")["churn_probability"]
    .agg(
        customer_count="count",
        churn_rate="mean"
    )
    .reset_index()
)

coordinated_deterioration_analysis["churn_rate_pct"] = (
    coordinated_deterioration_analysis["churn_rate"] * 100
)

display(coordinated_deterioration_analysis)

,coordinated_deterioration_count,customer_count,churn_rate,churn_rate_pct
0,0,16246,0.059153,5.915302
1,1,10642,0.048393,4.839316
2,2,8529,0.049478,4.947825
3,3,9071,0.079925,7.992504
4,4,10186,0.069900,6.989986
5,5,15325,0.247765,24.776509


In [52]:
# Count the number of core behavioural dimensions that show
# continuous deterioration from June through August.

persistent_deterioration_features = [
    "arpu_persistent_decline",
    "recharge_amount_persistent_decline",
    "recharge_count_persistent_decline",
    "outgoing_mou_persistent_decline",
    "incoming_mou_persistent_decline"
]

df_prepared["persistent_deterioration_count"] = (
    df_prepared[persistent_deterioration_features]
    .sum(axis=1)
)

print(
    df_prepared["persistent_deterioration_count"]
    .value_counts()
    .sort_index()
)

persistent_deterioration_count
0    38615
1    14270
2     7091
3     4788
4     3177
5     2058
Name: count, dtype: int64


In [53]:
persistent_deterioration_analysis = (
    df_prepared
    .groupby("persistent_deterioration_count")["churn_probability"]
    .agg(
        customer_count="count",
        churn_rate="mean"
    )
    .reset_index()
)

persistent_deterioration_analysis["churn_rate_pct"] = (
    persistent_deterioration_analysis["churn_rate"] * 100
)

display(persistent_deterioration_analysis)

,persistent_deterioration_count,customer_count,churn_rate,churn_rate_pct
0,0,38615,0.076758,7.675774
1,1,14270,0.093623,9.362299
2,2,7091,0.099986,9.998590
3,3,4788,0.142648,14.264829
4,4,3177,0.214353,21.435316
5,5,2058,0.368805,36.880466


## Recent Behavioural Deterioration

We have already captured two different behavioural concepts:

* **A. Coordinated deterioration** — how many behavioural dimensions deteriorated overall from June to August.
* **B. Persistent deterioration** — how many behavioural dimensions continuously declined from June to July to August.

Next, we capture **recent deterioration** — how many dimensions declined specifically from July to August.


In [54]:
# Count behavioural dimensions that deteriorated during the most recent
# observed month-to-month transition.

recent_deterioration_features = [
    "arpu_jul_aug_change",
    "recharge_amount_jul_aug_change",
    "recharge_count_jul_aug_change",
    "outgoing_mou_jul_aug_change",
    "incoming_mou_jul_aug_change"
]

df_prepared["recent_deterioration_count"] = (
    df_prepared[recent_deterioration_features] < 0
).sum(axis=1)

print(
    df_prepared["recent_deterioration_count"]
    .value_counts()
    .sort_index()
)

recent_deterioration_count
0    13901
1    12363
2    10228
3    10161
4    10595
5    12751
Name: count, dtype: int64


In [55]:
recent_deterioration_analysis = (
    df_prepared
    .groupby("recent_deterioration_count")["churn_probability"]
    .agg(
        customer_count="count",
        churn_rate="mean"
    )
    .reset_index()
)

recent_deterioration_analysis["churn_rate_pct"] = (
    recent_deterioration_analysis["churn_rate"] * 100
)

display(recent_deterioration_analysis)

,recent_deterioration_count,customer_count,churn_rate,churn_rate_pct
0,0,13901,0.084167,8.416661
1,1,12363,0.087034,8.703389
2,2,10228,0.062182,6.218224
3,3,10161,0.077945,7.794508
4,4,10595,0.075507,7.550731
5,5,12751,0.208454,20.845424


## Final Feature Consolidation

We have now captured three different behavioural concepts:

* **A. Coordinated deterioration** — breadth of overall decline
* **B. Persistent deterioration** — continuity of decline
* **C. Recent deterioration** — latest behavioural decline

Before moving to modelling, we will perform final checks on the prepared dataset and ensure that the engineered features are consistent and ready for the next stage.


## Final Sanity Checks

Before saving the prepared dataset, we verify that:

* no missing values remain,
* the target is unchanged,
* customer IDs remain unique,
* and the engineered features contain valid values.


In [56]:
# Verify missing values
remaining_missing = df_prepared.isna().sum().sum()

# Verify target distribution
target_distribution = df_prepared["churn_probability"].value_counts()

# Verify customer ID uniqueness
duplicate_ids = df_prepared["id"].duplicated().sum()

# Verify infinite values
infinite_values = np.isinf(
    df_prepared.select_dtypes(include=np.number)
).sum().sum()

print("Final Sanity Check")
print("-" * 40)

print(f"Total missing values : {remaining_missing}")
print(f"Duplicate customer IDs: {duplicate_ids}")
print(f"Infinite values      : {infinite_values}")

print("\nTarget distribution:")
print(target_distribution)

Final Sanity Check
----------------------------------------
Total missing values : 0
Duplicate customer IDs: 0
Infinite values      : 0

Target distribution:
churn_probability
0    62867
1     7132
Name: count, dtype: int64


In [57]:
print(f"\nFinal dataset shape: {df_prepared.shape}")

print("\nSample of engineered features:")
display(
    df_prepared[
        [
            "coordinated_deterioration_count",
            "persistent_deterioration_count",
            "recent_deterioration_count"
        ]
    ].head()
)


Final dataset shape: (69999, 191)

Sample of engineered features:


,coordinated_deterioration_count,persistent_deterioration_count,recent_deterioration_count
0,4,1,4
1,0,0,3
2,4,0,4
3,4,0,4
4,4,4,4


In [ ]:
# Separate remaining missing values by their semantic role.
# Date fields retain missingness because absence of a date can represent
# absence of the corresponding activity.

date_features = [
    col for col in df_prepared.columns
    if col.startswith("date_of_last_rech")
]

remaining_missing = (
    df_prepared.isna().sum()
    .loc[lambda x: x > 0]
    .sort_values(ascending=False)
)

print(f"Remaining features with missing values: {len(remaining_missing)}")
display(remaining_missing)

Remaining features with missing values: 93


date_of_last_rech_data_6    52431
date_of_last_rech_data_7    52134
date_of_last_rech_data_8    51582
roam_ic_mou_8                3703
onnet_mou_8                  3703
                            ...  
spl_ic_mou_7                 2687
ic_others_7                  2687
date_of_last_rech_8          2461
date_of_last_rech_7          1234
date_of_last_rech_6          1101
Length: 93, dtype: int64

## Save Prepared Dataset

The prepared dataset has passed the final consistency checks.

It is now saved separately from the raw data so that the original dataset remains unchanged and the preparation process can be reproduced.


In [59]:
from pathlib import Path

# Define the output location for the model-ready dataset.
PROCESSED_PATH = (
    Path("../data/processed")
    / "telecom_churn_prepared.csv"
)

# Save the prepared dataset without writing the DataFrame index as a column.
df_prepared.to_csv(PROCESSED_PATH, index=False)

# Confirm the output location and final dataset dimensions.
print(f"Prepared dataset saved to: {PROCESSED_PATH.resolve()}")
print(f"Final shape: {df_prepared.shape}")

Prepared dataset saved to: E:\Churn Prediction\data\processed\telecom_churn_prepared.csv
Final shape: (69999, 191)


## Conclusion

In this notebook, the raw telecom churn dataset was prepared for machine
learning while preserving its behavioural and temporal information.

The preparation included data-quality auditing, removal of non-informative
metadata, investigation and treatment of structural missing values, and
conversion of meaningful missing states into appropriate activity indicators.

Temporal and behavioural features were then created to capture three
different concepts:

- Coordinated deterioration
- Persistent deterioration
- Recent deterioration

The final dataset contains 69,999 customers and 191 features, with no
remaining missing values, duplicate customer IDs, or infinite values.

The prepared dataset has been saved and is ready for the model development
and evaluation stage.